### Construct a model 

COBAHH.py

In [ ]:
import brian2 as b2
import numpy as np

from neuroseq.b2_model import B2Model

In [ ]:
b2.seed(42)
# Parameters
area = 20000 * b2.umetre**2
Cm = (1 * b2.ufarad * b2.cm**-2) * area
gl = (5e-5 * b2.siemens * b2.cm**-2) * area

El = -60 * b2.mV
EK = -90 * b2.mV
ENa = 50 * b2.mV
g_na = (100 * b2.msiemens * b2.cm**-2) * area
g_kd = (30 * b2.msiemens * b2.cm**-2) * area

VT = -63 * b2.mV

# Time constants
taue = 5 * b2.ms
taui = 10 * b2.ms
# Reversal potentials
Ee = 0 * b2.mV
Ei = -80 * b2.mV
we = 6 * b2.nS  # excitatory synaptic weight
wi = 67 * b2.nS  # inhibitory synaptic weight

# The model
eqs = b2.Equations("""
dv/dt = (gl*(El-v)+ge*(Ee-v)+gi*(Ei-v)-
         g_na*(m*m*m)*h*(v-ENa)-
         g_kd*(n*n*n*n)*(v-EK))/Cm : volt
dm/dt = alpha_m*(1-m)-beta_m*m : 1
dn/dt = alpha_n*(1-n)-beta_n*n : 1
dh/dt = alpha_h*(1-h)-beta_h*h : 1
dge/dt = -ge*(1./taue) : siemens
dgi/dt = -gi*(1./taui) : siemens
alpha_m = 0.32*(mV**-1)*4*mV/exprel((13*mV-v+VT)/(4*mV))/ms : Hz
beta_m = 0.28*(mV**-1)*5*mV/exprel((v-VT-40*mV)/(5*mV))/ms : Hz
alpha_h = 0.128*exp((17*mV-v+VT)/(18*mV))/ms : Hz
beta_h = 4./(1+exp((40*mV-v+VT)/(5*mV)))/ms : Hz
alpha_n = 0.032*(mV**-1)*5*mV/exprel((15*mV-v+VT)/(5*mV))/ms : Hz
beta_n = .5*exp((10*mV-v+VT)/(40*mV))/ms : Hz
""")

P = b2.NeuronGroup(
    4000,
    model=eqs,
    threshold="v>-20*mV",
    refractory=3 * b2.ms,
    method="exponential_euler",
)
Pe = P[:3200]
Pi = P[3200:]
Ce = b2.Synapses(Pe, P, on_pre="ge+=we")
Ci = b2.Synapses(Pi, P, on_pre="gi+=wi")
Ce.connect(p=0.02)
Ci.connect(p=0.02)

# Initialization
P.v = "El + (randn() * 5 - 5)*mV"
P.ge = "(randn() * 1.5 + 4) * 10.*nS"
P.gi = "(randn() * 12 + 20) * 10.*nS"

# Record a few traces

Traces_v = b2.StateMonitor(P, "v", record=True)
# Traces_n = b2.StateMonitor(P, 'n',record=True)

# Traces_alpha_beta = b2.StateMonitor(P, ['alpha_m','beta_m','alpha_h','beta_h','alpha_n','beta_n'],record=True)

# trace = b2.StateMonitor(P, 'v', record=[1, 10, 100])

b2.run(1 * b2.second, report="text")

In [ ]:
b2.plot(Traces_v.t / b2.ms, Traces_v[1].v / b2.mV)
b2.plot(Traces_v.t / b2.ms, Traces_v[10].v / b2.mV)
b2.plot(Traces_v.t / b2.ms, Traces_v[100].v / b2.mV)
b2.xlabel("t (ms)")
b2.ylabel("v (mV)")
b2.xlim([0, 200])
b2.show()

In [ ]:
# b2m class implementation

model = B2Model(seed=42)

tfinal = 1000 * b2.ms

nneurons = 4000

model.define_parameter("area", 20000 * b2.umetre**2)
model.define_parameter("Cm", (1 * b2.ufarad * b2.cm**-2) * model.parameters["area"])
model.define_parameter("gl", (5e-5 * b2.siemens * b2.cm**-2) * model.parameters["area"])

model.define_parameter("El", -60 * b2.mV)
model.define_parameter("EK", -90 * b2.mV)
model.define_parameter("ENa", 50 * b2.mV)
model.define_parameter(
    "g_na", (100 * b2.msiemens * b2.cm**-2) * model.parameters["area"]
)
model.define_parameter(
    "g_kd", (30 * b2.msiemens * b2.cm**-2) * model.parameters["area"]
)
model.define_parameter("VT", -63 * b2.mV)
# Time constants
model.define_parameter("taue", 5 * b2.ms)
model.define_parameter("taui", 10 * b2.ms)
# # Reversal potentials
model.define_parameter("Ee", 0 * b2.mV)
model.define_parameter("Ei", -80 * b2.mV)
model.define_parameter("we", 6 * b2.nS)
model.define_parameter("wi", 67 * b2.nS)

opto_units = 1 * b2.mwatt / b2.um2

model.define_parameter("opto_units", opto_units)


i_stim = np.zeros((int(tfinal / b2.ms), nneurons), dtype="float")
i_stim[:, :] = 0
i_stim[200:300, 0] = 1
I_external = b2.TimedArray(i_stim * b2.pA, dt=1 * b2.ms)

exc_stim = np.zeros((int(tfinal / b2.ms), nneurons), dtype="float")
exc_stim[:, :] = 0
exc_stim[0:200, 0] = 1
Exc_stimulation = b2.TimedArray(exc_stim * opto_units, dt=1 * b2.ms)

inh_stim = np.zeros((int(tfinal / b2.ms), nneurons), dtype="float")
inh_stim[:, :] = 0
inh_stim[300:900, 0] = 1
Inh_stimulation = b2.TimedArray(inh_stim * opto_units, dt=1 * b2.ms)

# +sigma*sqrt(2/tau)*xi

model.define_parameter("I_external", I_external)
model.define_parameter("Exc_stimulation", Exc_stimulation)
model.define_parameter("Inh_stimulation", Inh_stimulation)


eqs = """
dv/dt = (geopto*(Eeopto-v)+giopto*(Eiopto-v)+gl*(El-v)-
         g_na*(m*m*m)*h*(v-ENa)-
         g_kd*(n*n*n*n)*(v-EK)+Iext)/Cm : volt
Iext = I_external(t,i) : amp
geopto = g_exc * Exc_stimulation(t, i)/opto_units: siemens
giopto = g_inh * Inh_stimulation(t, i)/opto_units: siemens
dm/dt = alpha_m*(1-m)-beta_m*m : 1
dn/dt = alpha_n*(1-n)-beta_n*n : 1
dh/dt = alpha_h*(1-h)-beta_h*h : 1
dge/dt = -ge*(1./taue) : siemens
dgi/dt = -gi*(1./taui) : siemens
alpha_m = 0.32*(mV**-1)*4*mV/exprel((13*mV-v+VT)/(4*mV))/ms : Hz
beta_m = 0.28*(mV**-1)*5*mV/exprel((v-VT-40*mV)/(5*mV))/ms : Hz
alpha_h = 0.128*exp((17*mV-v+VT)/(18*mV))/ms : Hz
beta_h = 4./(1+exp((40*mV-v+VT)/(5*mV)))/ms : Hz
alpha_n = 0.032*(mV**-1)*5*mV/exprel((15*mV-v+VT)/(5*mV))/ms : Hz
beta_n = .5*exp((10*mV-v+VT)/(40*mV))/ms : Hz
"""

# neuron group
model.define_neuronnumber("P", nneurons)
model.define_solver_method("P", "exponential_euler")
model.define_equation("P", eqs)
model.define_threshold("P", "v>-20*mV")
# model.define_reset("neuron", "v=-65*mV")
model.define_refractory("P", 2 * b2.ms)
model.define_dt("P", 0.1 * b2.ms)
model._assemble_neurongroup("P")


model.define_neuronsubgroup_target("Pe", "P")
model.define_neuronsubgroup_idxs("Pe", (0, 24))

model.define_neuronsubgroup_target("Pi", "P")
model.define_neuronsubgroup_idxs("Pi", (0, 24))

model._assemble_neuronsubgroup("Pi")
model._assemble_neuronsubgroup("Pe")

model.define_synapse_connection("Ce", "p=0.02")
model.define_synapse_connection_probability("Ce", 0.02)
model.define_synapse_connection_probability("Ci", 0.02)
# model.define_synapse_origin("Ce", "Pe")
# model.define_synapse_target("Ce", "Pi")


Ce = b2.Synapses(Pe, P, on_pre="ge+=we")
Ci = b2.Synapses(Pi, P, on_pre="gi+=wi")
Ce.connect(p=0.02)
Ci.connect(p=0.02)

# initial conditions
model.define_init("P", "v", "El + (randn() * 5 - 5)*mV")
model.define_init("P", "ge", "(randn() * 1.5 + 4) * 10.*nS")
model.define_init("P", "gi", "(randn() * 12 + 20) * 10.*nS")


# monitors
model.define_monitor_source("trace", "neuron")
model.define_monitor_type("trace", "StateMonitor")
model.define_monitor_variable("trace", True)
# model.define_monitor_variable("trace", "Istim")
model._assemble_monitor("trace")

model.define_monitor_source("mon", "neuron")
model.define_monitor_type("mon", "SpikeMonitor")
model._assemble_monitor("mon")

model.assemble_network()